# Phase 0.6 follow-ups on Colab — F2 (camera embedding) and F3 (idle-trimmed data)

Companion to `Details/phase06_camera_ablation_training.md` §8. Everything is pulled from the Hub;
nothing local is needed.

| | What it tests | Runs |
|---|---|---|
| **F2** | Is `top+wrist`'s reach deficit an *arbitration* problem? ACT gives both cameras identical positional embeddings, so the model can only tell the streams apart by appearance. Add a learned per-camera identity vector and retrain. | 1 × 60k (`both`) |
| **F3** | Does idle-trimming the training data let short action chunks work? The untrimmed set forced `n_action_steps=100`, i.e. ~9 observations per 30 s episode, which blunts any camera ablation. | 3 × 60k (`top`, `wrist`, `both`) |

**The recipe is identical to the 60k baselines** — same 45/5 balanced holdout, same seed 1000, same
`batch_size=8`, `eval_steps=5000`, fp32, no augmentation — so the new runs are directly comparable to
`act_top_s1000` / `act_wrist_s1000` / `act_both_s1000` in the same wandb project.

## Which Colab GPU?

**Pick L4.** The workload is small-convolution compute-bound, not memory-bound: ACT at batch 8 with two
cameras peaks at **3.5 GiB**, so an A100's 40 GB is pure waste, and a bigger batch buys nothing (measured
locally: throughput is flat at 58–60 samples/s from batch 8 to 32).

Estimates scaled from a measured RTX 5080 Laptop baseline (fp32: 5.3 it/s two-camera, 10.5 single):

| GPU | VRAM | est. it/s (2 cam) | F2 total | F3 total | F2+F3 |
|---|---|---|---|---|---|
| T4 | 16 GB | ~1.6 | ~10.5 h | ~21 h | **~32 h** — impractical, risks session limits |
| **L4** | **24 GB** | **~3.7** | **~4.5 h** | **~9 h** | **~13.5 h** ← recommended |
| A100 | 40 GB | ~6.4 | ~2.6 h | ~5.2 h | ~7.8 h, at ~2.5× the compute units per hour |

Roughly, per compute unit: L4 ≈ 65 units for both experiments, A100 ≈ 92, T4 ≈ 57 but spread over days.
**L4 is the best time-per-unit trade** and has ample VRAM. Check current Colab rates — they change.

> These are estimates. **The section 5 smoke test measures your actual throughput** and prints real
> projections before you commit to a long run. Measured on this account's runtime on 2026-09-22:
> **4.5 it/s two-camera**, i.e. F2 ≈ 3.7 h and F3 ≈ 7.5 h — better than the L4 estimate above.

**Session limits matter.** Each individual run fits in a typical Colab session, but F3's three runs do
not. Every run below pushes checkpoints to the Hub, so a disconnect is recoverable — see §9.

## How to run this

**Run all works**, and is the sensible way to do it if the smoke test is green. Active execution keeps a
Colab session alive, so the ~11 h programme (smoke ≈10 min + F2 ≈3.7 h + F3 ≈7.5 h) fits inside a paid
tier's VM lifetime. Before starting, check two things:

- **Compute units.** ~11 h at your GPU's rate — roughly 55 units on an L4, ~130 on an A100. Running out
  mid-run stops the VM.
- **The current session limit.** Paid tiers allow long sessions but the published cap changes; if yours
  is below ~12 h, run F2 and F3 in separate sessions. On Pro+, enable background execution so closing
  the tab does not end the run.

Safety rails are in place for an unattended run:

- The **§5 smoke test raises** if any check fails, so a bad token or config halts the queue in ten
  minutes instead of failing at hour four.
- **§7 skips** rebuilding the trimmed dataset if the repo already exists.
- **`use_camera_embed()` is called explicitly before each experiment**, and every training cell asserts
  the resulting parameter count. This matters: the F2 patch edits `modeling_act.py` *on disk*, so it
  applies to every later `lerobot-train` subprocess — without the explicit revert, F3 would silently
  train with the camera embedding and the two experiments would be confounded.

**Uploads happen throughout, not only at the end.** With `--save_checkpoint_to_hub`, every `save_freq`
(10k steps) pushes the checkpoint to `checkpoints/<step>/` in the run's model repo and tags it with the
step, so any checkpoint can be recovered with `--policy.pretrained_revision=<step>`. At the end,
`--policy.push_to_hub` additionally writes the final model plus its pre/post-processors to the repo root.
Six intermediate pushes and one final push per run — so a disconnect costs at most the steps since the
last save, and §9 resumes from the Hub into the same wandb run.

One thing Run all does not do: re-run the smoke test against the trimmed dataset. If you want that check,
run §5 once with `SMOKE_DATASET = TRIMMED` before starting.

## 1 — Runtime check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", name)
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > GPU (L4).")
if "T4" in name:
    print("\n!! T4 detected. F2+F3 would take ~32 h here. Runtime > Change runtime type > L4.")
elif "A100" in name:
    print("\nNote: A100 works but costs ~2.5x the compute units for ~1.7x the speed; L4 is the better trade.")

## 2 — Install

Installed from the **v0.6.1 source tree** rather than the wheel, because F2 patches `modeling_act.py`.
Takes a few minutes.

In [ ]:
!git clone --depth 1 --branch v0.6.1 https://github.com/huggingface/lerobot.git /content/lerobot
%pip install -q -e /content/lerobot
%pip install -q wandb
import lerobot; print("lerobot", lerobot.__version__)

> If `import lerobot` fails right after the install, restart the runtime (Runtime > Restart session) and re-run this cell — Colab caches the old module table.

## 3 — Authentication

No proxy here, unlike the local machine — but there is a Colab-specific trap.

**A Colab secret alone is not enough.** `huggingface_hub` resolves a token in this order
(`utils/_auth.py`): OIDC → `HF_TOKEN` env var → `~/.cache/huggingface/token` → **Google Colab secret**.
That last path calls `google.colab.userdata.get()`, which needs the notebook kernel's channel to the
Colab frontend. `lerobot-train` runs as a **subprocess**, where that channel does not exist — so with only
a secret set, `whoami()` works in the notebook while training authenticates as *nobody*: 401 on a private
dataset, 401 on `create_repo`.

The cell below materialises the token into both the environment and the token file, then **proves a
subprocess can authenticate** before you rely on it.

In [ ]:
import os, sys, subprocess
from huggingface_hub import get_token, login, whoami

# 1. Find a token: env var, then token file, then the Colab secret (kernel-only), else prompt.
tok = os.environ.get("HF_TOKEN") or get_token()
if not tok:
    from huggingface_hub import notebook_login
    notebook_login()
    tok = get_token()
assert tok, "no HF token found — add one as a Colab secret named HF_TOKEN, or run notebook_login()"

# 2. Materialise it so every child process inherits it. Both paths, deliberately:
os.environ["HF_TOKEN"] = tok                      # env var -> inherited by subprocess
login(token=tok, add_to_git_credential=False)     # writes ~/.cache/huggingface/token

# 3. The check that actually matters.
r = subprocess.run([sys.executable, "-c",
                    "from huggingface_hub import whoami; print(whoami()['name'])"],
                   capture_output=True, text=True)
assert r.returncode == 0, "a subprocess still cannot authenticate:\n" + r.stderr[-800:]
print("notebook user:", whoami()["name"], "| subprocess user:", r.stdout.strip())

# 4. Scope check. These runs CREATE new model repos, which a per-repo-scoped token cannot do.
a = whoami().get("auth", {}).get("accessToken", {})
role = a.get("role")
if role == "fineGrained":
    g = (a.get("fineGrained") or {}).get("global", [])
    print("token: fine-grained | global scopes:", g)
    if not any("repo.write" in s for s in g):
        print("\n!! This token has no GLOBAL repo-write scope. It can read repos it is explicitly scoped\n"
              "   to, but cannot create the new model repos these runs push to — you will get a 401 on\n"
              "   create_repo at the first save_freq.\n"
              "   Fix: huggingface.co/settings/tokens -> edit this token -> tick\n"
              "   'Write access to contents/settings of all repos'. Or use a classic token with role Write.")
else:
    print("token role:", role, "('write' can create repos, 'read' cannot)")

In [ ]:
import os, sys, subprocess, wandb

# Same subprocess trap as HF: a Colab secret is readable only from the kernel. wandb.login() also
# persists to ~/.netrc, which children do read — set both.
key = os.environ.get("WANDB_API_KEY")
if not key:
    try:
        from google.colab import userdata
        key = userdata.get("WANDB_API_KEY")
    except Exception:
        key = None
if key:
    os.environ["WANDB_API_KEY"] = key
    wandb.login(key=key)
else:
    wandb.login()                                  # interactive; key from https://wandb.ai/authorize
    os.environ["WANDB_API_KEY"] = wandb.api.api_key

r = subprocess.run([sys.executable, "-c", "import wandb; print(bool(wandb.api.api_key))"],
                   capture_output=True, text=True)
assert r.stdout.strip() == "True", "a subprocess cannot see the wandb key:\n" + r.stderr[-500:]
print("wandb ok in notebook and subprocess")

## 4 — Shared configuration

The holdout is the fiddly part and must match the baselines exactly. The 50 episodes were recorded one
position at a time (`ep 0-4 = P1 … 45-49 = P10`), so lerobot's default "last N episodes" split would hold
out **all of P10** and remove that position from training. Reordering `--dataset.episodes` puts a balanced
holdout in the tail instead: the last episode of P2, P4, P6, P8, P10.

In [ ]:
HF_USER = "HALDijkstraaa"
DATASET = f"{HF_USER}/so101_toolkit_cylinder_20260917_165544"
TRIMMED = f"{DATASET}_trimmed"                  # built in section 7, or pushed from the laptop
PROJECT = "phase06-camera-ablation"             # same project as the baselines, so curves overlay
SEED, STEPS, BATCH = 1000, 60_000, 8

HOLDOUT  = [9, 19, 29, 39, 49]
EPISODES = [e for e in range(50) if e not in HOLDOUT] + HOLDOUT

STATE = "'observation.state': {'type': 'STATE', 'shape': [6]}"
CAM   = lambda c: f"'observation.images.{c}': {{'type': 'VISUAL', 'shape': [3, 480, 640]}}"
FEATS = {"top":   f"{{{STATE}, {CAM('top')}}}",
         "wrist": f"{{{STATE}, {CAM('wrist')}}}",
         "both":  f"{{{STATE}, {CAM('top')}, {CAM('wrist')}}}"}

# Checkpoints go to the Hub as well as to disk, so a Colab disconnect is recoverable (section 9).
def train_cmd(cond, job, dataset=DATASET, steps=STEPS, extra=()):
    return ["lerobot-train",
        f"--dataset.repo_id={dataset}",
        f"--dataset.episodes={EPISODES}",
        "--dataset.eval_split=0.1", "--eval_steps=5000",
        "--policy.type=act", "--policy.device=cuda",
        f"--policy.input_features={FEATS[cond]}",
        "--policy.push_to_hub=true", f"--policy.repo_id={HF_USER}/{job}", "--policy.private=true",
        "--save_checkpoint_to_hub=true",
        f"--batch_size={BATCH}", f"--steps={steps}", "--num_workers=8", f"--seed={SEED}",
        "--save_freq=10000", "--log_freq=200",
        f"--output_dir=/content/outputs/{job}", f"--job_name={job}",
        "--wandb.enable=true", f"--wandb.project={PROJECT}", *extra]

import re, subprocess, sys, time
BASE_PARAMS = 51_597_190          # stock ACT with 2 cameras; +1024 when the F2 patch is applied

def run(cmd, expect_params=None):
    """Stream a training run. expect_params guards against training the wrong model variant."""
    t0, seen = time.time(), None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        m = re.search(r"num_learnable_params=(\d+)", line)
        if m: seen = int(m.group(1))
    p.wait()
    print(f"\n[exit {p.returncode} after {(time.time() - t0) / 3600:.2f} h]")
    if expect_params is not None and seen is not None and seen != expect_params:
        raise RuntimeError(f"wrong model variant: trained {seen:,} params, expected {expect_params:,}. "
                           "Check use_camera_embed() above this cell.")
    return p.returncode

print("holdout:", HOLDOUT, "| episode-list tail:", EPISODES[-6:])

## 5 — Smoke test + throughput (run this first, ≈8–12 min)

300 steps with the **real** configuration — the balanced split, periodic validation, wandb, and a
checkpoint pushed to the Hub — then automatic checks on all of it, plus your measured it/s and the
resulting projections for F2 and F3.

This is the cell that catches an auth or config mistake in ten minutes instead of at hour four. It is
deliberately not a stripped-down benchmark: the Hub push is the step that failed on the laptop, and with
`save_checkpoint_to_hub` the first push happens at `save_freq`, so a 300-step run proves it.

Most of the time is the one-time dataset download (~1 GB) and one 591 MB checkpoint push, not the 300
training steps. It writes to a throwaway repo you can delete afterwards.

In [ ]:
import re, shutil, subprocess, time
from huggingface_hub import HfApi

SMOKE_DATASET = DATASET          # switch to TRIMMED to also pre-verify the trimmed repo before F3
SMOKE_REPO    = f"{HF_USER}/act_smoke_delete_me"
JOB, N        = "smoke", 300

shutil.rmtree(f"/content/outputs/{JOB}", ignore_errors=True)
cmd = [c for c in train_cmd("both", JOB, dataset=SMOKE_DATASET, steps=N)
       if not c.startswith(("--policy.repo_id", "--save_freq", "--eval_steps"))]
cmd += [f"--policy.repo_id={SMOKE_REPO}", f"--save_freq={N}", f"--eval_steps={N // 2}"]

t0 = time.time()
r = subprocess.run(cmd, capture_output=True, text=True)
text = r.stdout + r.stderr
print(text[-2500:])

checks, rates = [], [float(x) for x in re.findall(r"([\d.]+)step/s", text)]
checks.append(("exit code 0", r.returncode == 0))
checks.append(("balanced split 45 train / 5 eval", "45 train, 5 eval" in text))
checks.append(("both cameras in input_features",
               "observation.images.top" in text and "observation.images.wrist" in text))
m = re.search(r"num_learnable_params=(\d+)", text)
checks.append((f"param count {m.group(1) if m else '?'} (51597190 baseline, +1024 if F2-patched)",
               m is not None and int(m.group(1)) in (51_597_190, 51_598_214)))
checks.append(("validation ran (eval_loss printed)", "eval_loss=" in text))
checks.append(("wandb run created", "wandb.ai" in text))
checks.append(("throughput parsed", bool(rates)))
try:
    files = HfApi().list_repo_files(SMOKE_REPO, repo_type="model")
    checks.append((f"checkpoint pushed to {SMOKE_REPO}",
                   any(f"checkpoints/{N:06d}/" in f for f in files)))
except Exception as e:
    checks.append((f"checkpoint pushed to {SMOKE_REPO} ({type(e).__name__})", False))

print("\n" + "=" * 62)
for label, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {label}")
print("=" * 62)

if rates:
    r2 = sorted(rates)[len(rates) // 2]          # median progress-bar rate, two cameras
    r1 = r2 * 1.95                                # one camera is ~2x (locally 134 vs 72 ms/step)
    print(f"\nmeasured on this GPU: {r2:.1f} it/s two-camera, ~{r1:.1f} it/s single-camera")
    print(f"  F2  (both, 60k)            : {STEPS / r2 / 3600:.1f} h")
    print(f"  F3  (top + wrist + both)   : {STEPS / r1 / 3600:.1f} + {STEPS / r1 / 3600:.1f} + "
          f"{STEPS / r2 / 3600:.1f} = {(2 * STEPS / r1 + STEPS / r2) / 3600:.1f} h")
    print(f"  add ~12 min per run for the 12 validation passes")
print(f"\nsmoke run wall clock: {(time.time() - t0) / 60:.0f} min "
      f"(mostly the dataset download and the checkpoint push)")

if all(ok for _, ok in checks):
    print("\nAll green — safe to start the real runs.")
    print(f"Clean up when convenient:  HfApi().delete_repo('{SMOKE_REPO}', repo_type='model')")
else:
    raise RuntimeError("smoke test failed — see the FAIL lines above. Fix before starting a long run.")
shutil.rmtree(f"/content/outputs/{JOB}", ignore_errors=True)

---
# F2 — a learned per-camera identity embedding

In lerobot's ACT every camera goes through **one shared ResNet18**, and `encoder_cam_feat_pos_embed` is a
2D sinusoidal embedding of the feature map's H×W — so `top`'s 300 tokens and `wrist`'s 300 tokens get
**identical positional embeddings**. The model must infer which camera a token came from purely from
appearance, then learn phase-dependent trust, from 45 demonstrations.

The patch adds `nn.Embedding(n_cameras, dim_model)` and adds the right vector to each camera's block.
**Zero-initialised**, so at step 0 the model is exactly upstream ACT — it can only add capacity, never
perturb the starting point. Cost: 2 × 512 = **1024 parameters**.

*(Same code as `scripts/patch_act_camera_embed.py` in the repo. Verified locally: params land at
51,597,190 + 1,024 and a forward pass runs.)*

In [ ]:
import shutil, subprocess, sys, textwrap
from pathlib import Path
import lerobot.policies.act.modeling_act as _m

BASE_PARAMS = globals().get("BASE_PARAMS", 51_597_190)   # also set in section 4
MODELING = Path(_m.__file__)
PRISTINE = MODELING.with_suffix(".py.pristine")
if not PRISTINE.exists():                      # keep an untouched copy the first time we are called
    shutil.copy(MODELING, PRISTINE)

A_OLD = (
    "        if self.config.image_features:\n"
    "            self.encoder_img_feat_input_proj = nn.Conv2d(\n"
    "                backbone_model.fc.in_features, config.dim_model, kernel_size=1\n"
    "            )\n")
A_NEW = A_OLD + (
    "            # F2: learned identity vector per camera, zero-init so step 0 == upstream ACT.\n"
    "            self.camera_id_embed = nn.Embedding(len(self.config.image_features), config.dim_model)\n"
    "            nn.init.zeros_(self.camera_id_embed.weight)\n")
B_OLD = (
    "            for img in batch[OBS_IMAGES]:\n"
    '                cam_features = self.backbone(img)["feature_map"]\n'
    "                cam_pos_embed = self.encoder_cam_feat_pos_embed(cam_features).to(dtype=cam_features.dtype)\n"
    "                cam_features = self.encoder_img_feat_input_proj(cam_features)\n")
B_NEW = (
    "            for cam_idx, img in enumerate(batch[OBS_IMAGES]):\n"
    '                cam_features = self.backbone(img)["feature_map"]\n'
    "                cam_pos_embed = self.encoder_cam_feat_pos_embed(cam_features).to(dtype=cam_features.dtype)\n"
    "                cam_features = self.encoder_img_feat_input_proj(cam_features)\n"
    "                # F2: tag the block with which camera it came from (broadcasts over h, w)\n"
    "                cam_features = cam_features + self.camera_id_embed.weight[cam_idx].view(1, -1, 1, 1)\n")

_CHECK = textwrap.dedent("""
    from lerobot.policies.act.configuration_act import ACTConfig
    from lerobot.policies.act.modeling_act import ACTPolicy
    from lerobot.configs.types import FeatureType, PolicyFeature as PF
    cfg = ACTConfig(device="cpu")
    cfg.input_features = {"observation.state": PF(type=FeatureType.STATE, shape=(6,)),
                          "observation.images.top": PF(type=FeatureType.VISUAL, shape=(3, 480, 640)),
                          "observation.images.wrist": PF(type=FeatureType.VISUAL, shape=(3, 480, 640))}
    cfg.output_features = {"action": PF(type=FeatureType.ACTION, shape=(6,))}
    p = ACTPolicy(cfg)
    print(sum(x.numel() for x in p.parameters()),
          bool(getattr(p.model, "camera_id_embed", None) is not None
               and (p.model.camera_id_embed.weight == 0).all()))
""")

def use_camera_embed(enabled: bool) -> int:
    """Switch the installed ACT between stock and F2-patched, and return the resulting param count.

    The patch edits modeling_act.py on disk, so it applies to EVERY later lerobot-train subprocess.
    F3 must run against stock ACT or idle-trimming is confounded with the camera embedding — hence an
    explicit call before each experiment rather than a one-way patch.
    """
    shutil.copy(PRISTINE, MODELING)                     # always start from the untouched file
    if enabled:
        s = PRISTINE.read_text()
        for old, new, where in ((A_OLD, A_NEW, "__init__"), (B_OLD, B_NEW, "forward")):
            assert s.count(old) == 1, f"anchor not found exactly once in {where} - version mismatch"
            s = s.replace(old, new)
        MODELING.write_text(s)
    r = subprocess.run([sys.executable, "-c", _CHECK], capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-1500:]
    n, zero = r.stdout.split()
    n = int(n)
    want = BASE_PARAMS + 1024 if enabled else BASE_PARAMS
    assert n == want, f"expected {want:,} params, got {n:,}"
    if enabled:
        assert zero == "True", "camera embedding must start at zero"
    print(f"ACT is now {'F2-patched' if enabled else 'stock'}: {n:,} params"
          f"{' (camera_id_embed zero-init)' if enabled else ''}")
    return n

print("use_camera_embed(True) before F2, use_camera_embed(False) before F3.")

### Enable the patch for F2

Verified in a subprocess — the same way `lerobot-train` will load the file.

In [ ]:
use_camera_embed(True)

### F2 training run

In [ ]:
run(train_cmd("both", f"act_both_s{SEED}_camemb"), expect_params=BASE_PARAMS + 1024)

---
# F3 — retrain on idle-trimmed data

Training episodes were never idle-trimmed: the arm sits still for a **median of 74 frames (2.5 s)** before
it first moves, and 50/50 episodes idle longer than 25 frames. So ACT at the home pose predicts a chunk
that starts with "stay still" in every episode it learned from, and any short action chunk stalls. Only
the full 100-step chunk clears the lead-in — which costs re-observation rate, and with it the sensitivity
of the whole camera ablation.

> **Faster: build the trimmed set on the laptop instead.** The trim is pure CPU — ~40 min on the 24-core
> machine versus 1–1.5 h of a paid GPU session here:
> ```
> python scripts/make_trimmed_dataset.py <src_repo> <dst_repo> ~/trimmed
> ```
> then push `~/trimmed` to the Hub and skip to section 8.

## 7 — Build the trimmed dataset (one-off; skip if it is already on the Hub)

In [ ]:
from huggingface_hub import HfApi

# Skip entirely if it is already on the Hub — rebuilding costs 1-1.5 h of a paid GPU session doing
# pure CPU work, and would overwrite a dataset the training runs may already be using.
if HfApi().repo_exists(TRIMMED, repo_type="dataset"):
    print(f"{TRIMMED} already exists on the Hub — skipping the rebuild.")
else:
    # Same logic as scripts/make_trimmed_dataset.py. Verified locally on a 3-episode set: parquet rows ==
    # decoded video frames for both cameras, and every trimmed episode already shows motion in its first
    # 30 frames. Roughly 13 frames/s on a 24-core box, slower here.
    import glob, shutil, time, numpy as np, pandas as pd, torch
    from pathlib import Path
    from lerobot.datasets.lerobot_dataset import LeRobotDataset

    THRESH, PAD, ROOT = 2.0, 5, Path("/content/trimmed")
    t0 = time.time()
    src = LeRobotDataset(DATASET)
    print(f"source: {src.num_episodes} episodes, {src.num_frames} frames")
    if ROOT.exists(): shutil.rmtree(ROOT)

    # States from the parquet — reading them through src[i] would decode both videos for every frame.
    raw = pd.concat([pd.read_parquet(f) for f in
                     sorted(glob.glob(str(Path(src.root) / "data/chunk-*/*.parquet")))]).sort_values("index")
    states = np.stack(raw["observation.state"].to_numpy())

    feats = {k: v for k, v in src.features.items()
             if k not in ("index", "episode_index", "frame_index", "timestamp", "task_index")}
    dst = LeRobotDataset.create(TRIMMED, fps=src.fps, features=feats, root=ROOT,
                                robot_type=src.meta.robot_type, use_videos=True)
    img_keys = [k for k in feats if k.startswith("observation.images")]

    kept = 0
    for ep in range(src.num_episodes):
        lo = int(src.meta.episodes["dataset_from_index"][ep])
        hi = int(src.meta.episodes["dataset_to_index"][ep])
        st = states[lo:hi]
        dev0 = np.abs(st - st[0]).max(axis=1)        # departure from the start pose
        dev1 = np.abs(st - st[-1]).max(axis=1)       # departure from the final pose
        a = max(0, (int(np.argmax(dev0 > THRESH)) if (dev0 > THRESH).any() else 0) - PAD)
        back = np.where(dev1 > THRESH)[0]
        b = min(len(st), (int(back[-1]) + 1 if len(back) else len(st)) + PAD)
        for i in range(lo + a, lo + b):              # only kept frames get decoded
            item = src[i]
            frame = {"task": item["task"]}
            for k in feats:
                v = item[k]
                if k in img_keys:                     # CHW float [0,1] -> HWC uint8
                    v = (v.permute(1, 2, 0).numpy() * 255).round().clip(0, 255).astype(np.uint8)
                elif isinstance(v, torch.Tensor):
                    v = v.numpy()
                frame[k] = v
            dst.add_frame(frame)
        dst.save_episode()
        kept += b - a
        print(f"  ep{ep:02d}: {hi - lo:4d} -> {b - a:4d}  (cut {a} lead-in, {len(st) - b} tail)", flush=True)

    print(f"kept {kept}/{src.num_frames} ({100 * kept / src.num_frames:.0f}%) in {(time.time() - t0) / 60:.0f} min")
    dst.push_to_hub(private=True)
    print("pushed", TRIMMED)


## 8 — F3 training runs

Three runs, sequentially, one per cell so a disconnect only costs the run in flight. **Steps stay at
60k**, not epochs — the trimmed set has ~17% fewer frames, so these see slightly more passes than the
baselines. That is the right call for comparability: same optimisation budget, different data.

> Before the first F3 run, it is worth re-running the section 5 smoke test with `SMOKE_DATASET = TRIMMED` — it is the only thing that proves the trimmed repo downloads and splits 45/5 the same way.


**Revert to stock ACT first.** The F2 patch edits `modeling_act.py` on disk, so without this every F3 run would silently include the camera embedding and the two experiments would be confounded.

In [ ]:
use_camera_embed(False)   # F3 must use stock ACT, or it is confounded with F2

In [ ]:
run(train_cmd("top", f"act_top_s{SEED}_trim", dataset=TRIMMED), expect_params=BASE_PARAMS)

In [ ]:
run(train_cmd("wrist", f"act_wrist_s{SEED}_trim", dataset=TRIMMED), expect_params=BASE_PARAMS)

In [ ]:
run(train_cmd("both", f"act_both_s{SEED}_trim", dataset=TRIMMED), expect_params=BASE_PARAMS)

---
## 9 — Resuming after a disconnect

Every run pushes checkpoints to its Hub repo, so nothing is lost. lerobot downloads the highest-numbered
checkpoint into a fresh local dir and continues, keeping the same wandb run:

```python
run(["lerobot-train", f"--config_path={HF_USER}/act_both_s1000_camemb",
     "--resume=true", f"--steps={STEPS}", "--output_dir=/content/outputs/resumed"])
```

Step counter, optimizer state, episode order, holdout and seed all come back from the checkpoint; only
`--steps` is read from the CLI.

## 10 — What to compare

Everything lands in the **`phase06-camera-ablation`** wandb project next to the baselines, so `eval_loss`
overlays directly.

| Compare | Against | Reading |
|---|---|---|
| `act_both_s1000_camemb` | `act_both_s1000` | A lower held-out loss is *encouraging but not the answer* — the baseline's deficit was in reach behavior, which action L1 barely sees. The real test is the robot |
| `act_*_s1000_trim` | `act_*_s1000` | Expect a **higher** held-out loss: the trimmed set has no trivial "stay still" frames left to predict. That is not a regression |

**Both experiments end at the robot, not at the loss curve.** Pull the checkpoints with `hf download`,
then run the §8 retest protocol — interleaved against the relevant baseline in one session, scoring
*reached the cylinder* rather than success, and for F3 with a short `--policy.n_action_steps` (25), which
is the whole point of the trim.